# Claude Agent SDK — playground

A hands-on tour of the SDK, built around **evistream's actual use case**: pulling a
table out of a research paper.

Each lesson is one idea, one cell, and prints what it cost.

---

### Before you run anything

**These cells make real API calls and spend real money.** Cell 1 sets a hard
session cap (`SESSION_BUDGET_USD`) and every helper checks it before calling out.
Total for a full pass, lessons 1-6: roughly **$0.30-0.60**. Lesson 7 is free.
Lesson 8 is opt-in and costs ~$0.50 because it runs a real extraction.

**Kernel:** the `topics` conda env
(`/home/ubuntu/miniconda3/envs/topics/bin/python`) — that's where
`claude-agent-sdk` and the evistream backend both live.

> Why this exists: on 2026-08-05 an agentic extraction in production spent $1.90
> and returned zero rows because its budget cap was set below its actual cost, and
> the resulting failure looked like an *empty answer* to the pipeline, which then
> retried it. Lesson 6 reproduces that exact trap deliberately, cheaply.


## 0. Setup — env, budget guard, helpers


In [ ]:
import os, sys, json, asyncio, tempfile, textwrap
from pathlib import Path

REPO    = Path('/home/ubuntu/evistream')
BACKEND = REPO / 'backend'
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

# Production keys live in AWS Secrets Manager, not .env. This injects them
# into os.environ exactly the way the FastAPI app and Celery workers do.
os.environ.setdefault('AWS_SECRETS_NAME', 'evistream/production')
os.environ.setdefault('AWS_REGION', 'us-east-1')
from utils.secrets_loader import load_secrets
load_secrets()

import claude_agent_sdk as sdk
from claude_agent_sdk import (
    query, ClaudeAgentOptions, AssistantMessage, ResultMessage,
    TextBlock, ToolUseBlock, tool, create_sdk_mcp_server, ToolAnnotations,
    HookMatcher, AgentDefinition,
)

print('sdk            ', getattr(sdk, '__version__', '?'))
print('ANTHROPIC_API_KEY set:', bool(os.environ.get('ANTHROPIC_API_KEY')))

MODEL = 'claude-sonnet-5'   # bare id — the SDK does NOT take the litellm 'anthropic/' prefix


### Reload after a backend edit

The kernel caches `dspy_components.agentic_table` from first import. If someone
edits it on disk you keep running the old code — and the symptom is confusing:
`TypeError: Config.__init__() got an unexpected keyword argument ...`.

Run this cell after any change to the extractor. If it still misbehaves,
restart the kernel — only the `SPENT` tally is lost.


In [ ]:
import importlib
from dspy_components import agentic_table as A
A = importlib.reload(A)

cfg = A.Config()
print('module :', A.__file__)
print('knobs  :', [f.name for f in __import__('dataclasses').fields(cfg)
                   if f.name.startswith(('budget_', 'effort_'))])
print('caps   : session ${:.2f} | extraction ${:.2f}'.format(
    cfg.budget_usd_session_cap, cfg.budget_usd_extraction_cap))


In [ ]:
# ── Hard spend guard ─────────────────────────────────────────────────────────
# Every helper below refuses to call the API once this is reached. Raise it
# deliberately if you want to run more; don't remove it.
SESSION_BUDGET_USD = 2.00
SPENT = 0.0


def _charge(amount: float) -> None:
    global SPENT
    SPENT += amount or 0.0


def _check_budget() -> None:
    if SPENT >= SESSION_BUDGET_USD:
        raise RuntimeError(
            f'Session cap reached: ${SPENT:.4f} of ${SESSION_BUDGET_USD:.2f}. '
            f'Raise SESSION_BUDGET_USD if you really want to keep going.'
        )


async def run(prompt, options=None, show='summary'):
    """Run one session and return (messages, result).

    show='summary' -> one line per turn
    show='full'    -> every block, including tool inputs (this is the
                      'what is the agent actually doing' view)
    show='quiet'   -> nothing but the final cost line
    """
    _check_budget()
    msgs, result = [], None
    async for m in query(prompt=prompt, options=options):
        msgs.append(m)
        if isinstance(m, AssistantMessage) and show != 'quiet':
            for b in m.content:
                if isinstance(b, TextBlock) and b.text.strip():
                    txt = b.text.strip()
                    print('  SAY   ', txt if show == 'full' else txt[:160])
                elif isinstance(b, ToolUseBlock):
                    if show == 'full':
                        print('  CALL  ', b.name, json.dumps(b.input)[:300])
                    else:
                        print('  CALL  ', b.name)
        elif isinstance(m, ResultMessage):
            result = m
    if result is not None:
        _charge(getattr(result, 'total_cost_usd', 0.0) or 0.0)
        print(f"  == {result.subtype} | turns={result.num_turns} "
              f"| ${getattr(result,'total_cost_usd',0) or 0:.4f} "
              f"| session total ${SPENT:.4f}")
    return msgs, result


def scratch(**files) -> Path:
    """Temp dir the agent is allowed to see. Keeps Read/Grep from wandering."""
    d = Path(tempfile.mkdtemp(prefix='sdkplay_'))
    for name, body in files.items():
        (d / name.replace('__', '.')).write_text(body, encoding='utf-8')
    return d

print('helpers ready — cap ${:.2f}'.format(SESSION_BUDGET_USD))


## 1. The smallest possible session

`query()` is an async generator. You iterate it; the last thing it yields is a
`ResultMessage` carrying cost, turn count and the session id.

No tools here — this is just the model answering.


In [ ]:
opts = ClaudeAgentOptions(
    model=MODEL,
    system_prompt='You answer in one short sentence.',
    max_turns=2,
    max_budget_usd=0.10,      # per-session hard stop
    tools=[],                 # no built-ins at all
)
msgs, res = await run('What is a systematic review, in one sentence?', opts, show='full')

print()
print('message types seen:', [type(m).__name__ for m in msgs])
print('session_id        :', res.session_id)


## 2. Giving it tools — and watching it use them

This is the part you can't currently see in the evistream worker log.

`tools` controls what is **loaded into context**; `allowed_tools` controls what
may **run without asking**. `permission_mode='dontAsk'` means anything outside
the allowlist is denied outright rather than prompting (nothing is interactive
here, so a prompt would just hang).

Watch the `CALL` lines: that's the agent deciding to grep rather than guess.


In [ ]:
paper = scratch(**{'paper__md': textwrap.dedent('''
    # A trial of two mouthwashes

    ## Methods
    Patients were randomised to chlorhexidine or saline.

    ## Results
    | Arm            | n  | Plaque index at 6 weeks |
    |----------------|----|--------------------------|
    | Chlorhexidine  | 42 | 1.21 (0.30)              |
    | Saline         | 40 | 1.88 (0.41)              |

    Bleeding on probing was not measured in this study.
''')})
print('scratch dir:', paper)

opts = ClaudeAgentOptions(
    model=MODEL,
    cwd=str(paper),
    tools=['Read', 'Grep'],
    allowed_tools=['Read', 'Grep'],
    disallowed_tools=['Write', 'Edit', 'Bash', 'WebSearch', 'WebFetch'],
    permission_mode='dontAsk',
    setting_sources=[],        # do NOT load the repo's CLAUDE.md into this run
    max_turns=8,
    max_budget_usd=0.25,
)

_ = await run(
    'Read paper.md. How many patients were in each arm? '
    'Also: was bleeding on probing measured? Grep before you answer.',
    opts, show='full',
)


**Try changing:**
- drop `'Grep'` from `tools` — does it still answer the second question honestly?
- set `setting_sources=['project']` — the repo's CLAUDE.md (~4k tokens about
  *editing evistream*) gets loaded and starts competing with your instructions.
  This is why the production extractor passes `[]`.


## 3. Structured output — guaranteeing the shape

`output_format` constrains the final answer to a JSON schema. This is what makes
the extractor's `{value, source_text, status}` envelope reliable instead of hoped-for.

The parsed object arrives on `ResultMessage.structured_output`.


In [ ]:
ARM_SCHEMA = {
    'type': 'object',
    'properties': {
        'arms': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'arm_label': {'type': 'string'},
                    'n':         {'type': ['string', 'number']},
                    'source_text': {'type': 'string'},
                },
                'required': ['arm_label', 'n', 'source_text'],
                'additionalProperties': False,
            },
        },
    },
    'required': ['arms'],
    'additionalProperties': False,
}

opts = ClaudeAgentOptions(
    model=MODEL, cwd=str(paper),
    tools=['Read', 'Grep'], allowed_tools=['Read', 'Grep'],
    permission_mode='dontAsk', setting_sources=[],
    max_turns=6, max_budget_usd=0.25,
    output_format={'type': 'json_schema', 'schema': ARM_SCHEMA},
)

_, res = await run(
    'Read paper.md and return every treatment arm with its n. '
    'source_text must be copied verbatim from the file.',
    opts, show='summary',
)

out = getattr(res, 'structured_output', None)
print()
print(json.dumps(out, indent=2) if out else 'no structured_output on this SDK build')


## 4. Your own tool (in-process MCP)

`@tool` + `create_sdk_mcp_server` register a Python function the agent can call.
No separate server process — it runs in *this* kernel, so you can print from
inside the handler and watch it fire.

This is how the real extractor turns 'don't miss rows' from a request into a
recorded commitment: `commit_row_plan` is exactly this pattern.


In [ ]:
CALLS = []

@tool('register_rows',
      'Register the complete list of rows you found BEFORE extracting any values. '
      'Call exactly once.',
      {'type': 'object',
       'properties': {'rows': {'type': 'array', 'items': {'type': 'string'}},
                      'evidence': {'type': 'string'}},
       'required': ['rows', 'evidence']},
      annotations=ToolAnnotations(readOnlyHint=True))
async def register_rows(args):
    CALLS.append(args)
    print('    >>> handler ran in this kernel:', args['rows'], '|', args['evidence'][:60])
    return {'content': [{'type': 'text',
                         'text': f"Registered {len(args['rows'])} rows. "
                                 f'Your final answer must cover exactly these.'}]}

server = create_sdk_mcp_server(name='play', version='1.0.0', tools=[register_rows])

opts = ClaudeAgentOptions(
    model=MODEL, cwd=str(paper),
    tools=['Read', 'Grep'],
    mcp_servers={'play': server},
    allowed_tools=['Read', 'Grep', 'mcp__play__register_rows'],
    permission_mode='dontAsk', setting_sources=[],
    max_turns=8, max_budget_usd=0.25,
)

_ = await run(
    'Read paper.md. First call register_rows with every treatment arm you find, '
    'then tell me the plaque index for each.',
    opts, show='full',
)
print()
print('handler was called', len(CALLS), 'time(s)')


## 5. Hooks — observing and vetoing from outside the model

A hook runs in **your** process, costs no tokens, and can block a call before it
happens. Useful for auditing (what did it try?) and for enforcing order
(refuse X until Y has happened).


In [ ]:
SEEN = []

async def pre_tool(input_data, tool_use_id, context):
    name = (input_data or {}).get('tool_name', '?')
    SEEN.append(name)
    print('    [hook] about to call:', name)
    if name == 'Grep':
        # Veto, with a reason the model gets to read.
        return {'hookSpecificOutput': {
            'hookEventName': 'PreToolUse',
            'permissionDecision': 'deny',
            'permissionDecisionReason': 'Grep is disabled in this exercise — use Read.',
        }}
    return {}

opts = ClaudeAgentOptions(
    model=MODEL, cwd=str(paper),
    tools=['Read', 'Grep'], allowed_tools=['Read', 'Grep'],
    permission_mode='dontAsk', setting_sources=[],
    hooks={'PreToolUse': [HookMatcher(hooks=[pre_tool])]},
    max_turns=8, max_budget_usd=0.25,
)

_ = await run('Grep paper.md for "bleeding", then summarise the file.', opts, show='full')
print()
print('hook saw:', SEEN)


## 6. Cost controls — and the trap that bit us in production

Three dials:

| dial | what it does |
|---|---|
| `max_budget_usd` | hard stop; session ends with `subtype='error_max_budget_usd'` |
| `max_turns` | hard stop on turn count |
| `effort` | `low`..`max`; thinking bills as **output** tokens, so this is the big one |

The cell below sets a deliberately impossible budget so you can see the failure
shape. **This is the bug we shipped:** the extractor's cap was
`0.45 + 0.012 x expected_rows`, which for an 11-column table came to **$0.498** —
below what a 9-column table had already been measured to cost ($0.539). So it was
guaranteed to trip. Worse, the extractor turned that into an empty envelope, the
pipeline read *empty* as *retryable*, and re-ran the whole thing up to 3 times.
$1.90 for zero rows.

Two lessons worth carrying into the project:
1. A budget you expect to hit is not a safety net, it's a failure generator.
2. **How you report the failure matters as much as the failure.** Budget
   exhaustion is deterministic — retrying is pure waste.


In [ ]:
opts = ClaudeAgentOptions(
    model=MODEL, cwd=str(paper),
    tools=['Read', 'Grep'], allowed_tools=['Read', 'Grep'],
    permission_mode='dontAsk', setting_sources=[],
    effort='high',
    max_turns=10,
    max_budget_usd=0.02,      # far too small on purpose
)

msgs, res = await run(
    'Read paper.md and write an exhaustive critical appraisal of the trial.',
    opts, show='summary',
)
print()
print('subtype     :', res.subtype)
print('is_error    :', getattr(res, 'is_error', None))
print('structured  :', getattr(res, 'structured_output', None))
print()
print('^ note there is no usable answer. Whatever you return to your caller here',
      'decides whether it retries. Returning something that looks empty is what',
      'cost us $1.90.', sep='\n  ')


## 7. evistream's real extractor — free inspection

No API calls in this cell. It loads a real compiled form and shows you the three
things the production extractor derives from it.


In [ ]:
from dspy_components import agentic_table as A

SCHEMA = REPO / 'eval/studies/ablation/base_schemas/dynamic_6c8eecce_ContinuousOutcomesV2.json'
schema_def = json.loads(SCHEMA.read_text())
sig_def, field_def = A.pick_table_field(schema_def)

cols    = [c['field_name'] for c in field_def['subform_fields']]
anchors = field_def.get('anchor_columns') or []
print('field   :', field_def['name'])
print('columns :', len(cols), cols)
print('anchors :', anchors, '  <- row identity')
print('values  :', [c for c in cols if c not in anchors], '  <- what Stage 2 fills')
print()

out_schema = A.build_output_schema(field_def)
print('generated output schema, first level:', list(out_schema['properties']))
print()

brief = A.compose_field_brief(sig_def, field_def)
print('composed field brief (first 1200 chars):')
print('-' * 70)
print(brief[:1200])


### The knobs, and what they're currently set to


In [ ]:
cfg = A.Config()
for k in ['model', 'effort_small', 'effort_large', 'effort_row_switch',
          'budget_usd_base', 'budget_usd_per_row', 'budget_usd_cap',
          'envelope_precheck_max_rows', 'sample_value_cells',
          'max_repair_rounds', 'use_row_census_subagent', 'trace']:
    print(f'  {k:28s} {getattr(cfg, k)}')

print()
MEASURED_9x6 = 0.539   # real run, Artese 2015, 9 cols x 6 rows, no repair round
for n_cols in (9, 11, 18):
    exp_rows = 8 if n_cols <= 10 else 4
    budget = min(cfg.budget_usd_cap,
                 cfg.budget_usd_base + exp_rows * cfg.budget_usd_per_row)
    margin = budget - MEASURED_9x6
    print(f'{n_cols:>3} cols -> guessed rows={exp_rows}, budget=${budget:.3f}',
          f'| vs one measured 9x6 run (${MEASURED_9x6}): {margin:+.3f}')

print()
print('The guess drops from 8 rows to 4 as soon as a table exceeds 10 columns,')
print('so a WIDER table gets a SMALLER budget - backwards, since output volume is')
print('roughly rows x columns. The 9-col case survives on ~1% margin; the 11-col')
print('case starts below a cost we had already measured. That is the open bug.')


## 8. Run the real extractor on a real paper (opt-in)

Set `REALLY_RUN = True` to execute. This calls the same code path production uses.

The cell prints the budget it will run under **before** it runs — that was the
gap that made the production failure invisible: the cap was inherited silently
from module defaults and nobody could see it was set below the known cost.

`trace=True` writes a step-by-step JSONL (row plan, every tool call, quote checks,
repair rounds) to `/tmp/evistream_agentic_traces/` — worth having on a run you pay for.


In [ ]:
import dataclasses

REALLY_RUN = False   # <- flip to True to spend real money

# Budget made EXPLICIT. Previously this cell inherited agentic_table.CFG
# silently, and you could not see what cap the run would get. A cap is a
# runaway backstop, not a target — set it well above expected cost.
RUN_CFG = dataclasses.replace(
    A.CFG,
    budget_usd_base=0.80,
    budget_usd_per_cell=0.012,
    budget_usd_session_cap=2.00,
    budget_usd_extraction_cap=3.00,
    trace=True,          # writes the step-by-step JSONL
)

anchors  = set(field_def.get('anchor_columns') or [])
n_val    = len([c for c in field_def['subform_fields']
                if c['field_name'] not in anchors])
exp_rows = 10        # the extractor's own over-estimate
cells    = exp_rows * n_val
cap = min(RUN_CFG.budget_usd_session_cap,
          RUN_CFG.budget_usd_base + cells * RUN_CFG.budget_usd_per_cell)
eff = (RUN_CFG.effort_large if n_val > RUN_CFG.effort_col_switch
       else RUN_CFG.effort_small)
print(f'{n_val} value cols x {exp_rows} assumed rows = {cells} cells')
print(f'session cap ${cap:.3f} | extraction cap '
      f'${RUN_CFG.budget_usd_extraction_cap:.2f} | effort {eff}')
print('measured reference: 9col/6row run cost $0.539')

if not REALLY_RUN:
    print()
    print('skipped (REALLY_RUN is False)')
else:
    _check_budget()
    md_path = REPO / 'eval/sheets/markdown_perio/Artese 2015.md'
    res = await A.extract_table_from_markdown(
        markdown_content=md_path.read_text(),
        sig_def=sig_def, field_def=field_def,
        paper_hint=md_path.stem, cfg=RUN_CFG,
    )
    _charge(res.cost_usd)
    print()
    print(f'status       {res.status}')
    print(f'rows         emitted={res.n_rows} planned={res.plan_rows} '
          f'missing={res.rows_missing} withdrawn={res.rows_withdrawn}')
    print(f'repairs      {res.repair_rounds}   turns {res.num_turns}')
    print(f'quotes       checked={res.quotes_checked} '
          f'failed_inloop={res.quotes_failed_inloop} '
          f'failed_grounding={res.cells_failed_grounding}')
    print(f'provenance   table={res.cells_from_table} prose={res.cells_from_prose} '
          f'nr={res.cells_nr}')
    print(f'cost         ${res.cost_usd:.4f}  ({res.duration_ms/1000:.0f}s)')
    print(f'notes        {res.notes or "-"}')
    print(f'trace        {res.trace_path or "-"}')
    env = (res.envelope or {}).get(field_def['name'])
    if isinstance(env, dict) and isinstance(env.get('value'), list) and env['value']:
        print()
        print('first row:')
        print(json.dumps(env['value'][0], indent=2)[:900])


## Where this leaves the project

**Two bugs are still open** in `backend/dspy_components/agentic_table.py`
(the form has been reverted to `standard`, so nothing is armed):

1. **Budget doesn't scale with column count.** Lesson 7's last cell shows the cap
   landing below the measured cost. Output volume is roughly `rows x columns`, but
   the formula only counts rows.
2. **A budget stop is reported as an empty result**, which `validate_extraction_output`
   treats as retryable — so the most expensive failure mode is the one that repeats.
   Deterministic failures should be terminal.

**Questions worth poking at with this notebook:**

- How much does `effort` actually move cost and quality on *your* tables? Lesson 6's
  options block is the place to sweep it.
- Does `Grep` earn its place? Re-run lesson 2 without it and see whether the
  'was bleeding measured?' answer degrades — that's the honest-NR behaviour the whole
  design rests on.
- Is `register_rows` (lesson 4) doing real work, or would the model have found the
  same rows anyway? That's `commit_row_plan` in miniature, and it's the main claim
  of the agentic approach.
- What *should* happen when the budget trips mid-extraction — return the rows
  collected so far, or nothing? Lesson 6 is where to think about it.
